In [1]:
from google.colab import drive
import hashlib
import os

drive.mount('/content/drive')

# Cài gdown nếu chưa có để tải trực tiếp file từ Google Drive link của Task 10
!pip install -q gdown

os.makedirs("/content/task11_data", exist_ok=True)
timesnet_path = "/content/task11_data/demand_timesnet.parquet"
eval_path     = "/content/drive/MyDrive/PTLDKD/data/eval.parquet"

# Tải trực tiếp demand_timesnet.parquet bằng ID từ run_info.md của Task 10
if not os.path.exists(timesnet_path):
    !gdown --id 1f9db-U9f9C9OFS-V2EcmQv4FIU3bguR3 -O {timesnet_path}

def get_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

sha_timesnet = get_sha256(timesnet_path)
sha_eval     = get_sha256(eval_path)

EXPECTED_TIMESNET_SHA256 = "c34f4eafe087d225865a624ef24d213af566496541458c3075f6e596d9f1d694"

print(f"SHA-256 demand_timesnet.parquet : {sha_timesnet}")
print(f"SHA-256 eval.parquet            : {sha_eval}")

assert sha_timesnet == EXPECTED_TIMESNET_SHA256, "Lỗi: Mã SHA-256 của demand_timesnet.parquet không khớp Task 10!"
print("Xác nhận khớp 100% mã SHA-256 của demand_timesnet.parquet (81,562,029 bytes)!")

Mounted at /content/drive
/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1f9db-U9f9C9OFS-V2EcmQv4FIU3bguR3
To: /content/task11_data/demand_timesnet.parquet
100% 81.6M/81.6M [00:01<00:00, 63.2MB/s]
SHA-256 demand_timesnet.parquet : c34f4eafe087d225865a624ef24d213af566496541458c3075f6e596d9f1d694
SHA-256 eval.parquet            : 678caca15afed20239f2ea79208856fe18dad17fd444472c7d7e930edfa685bd
Xác nhận khớp 100% mã SHA-256 của demand_timesnet.parquet (81,562,029 bytes)!


In [2]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime, timezone

os.makedirs("/content/task11_outputs/full", exist_ok=True)

timesnet_path = "/content/task11_data/demand_timesnet.parquet"
eval_path     = "/content/drive/MyDrive/PTLDKD/data/eval.parquet"

# 1. Nạp dữ liệu tối ưu RAM (tuyệt đối không ghi đè sale_amount gốc)
cols_eval = ['store_id', 'product_id', 'dt', 'sale_amount', 'stock_hour6_22_cnt', 'discount', 'holiday_flag', 'precpt']
eval_df = pd.read_parquet(eval_path, columns=cols_eval)
eval_df['sale_amount_pred'] = np.nan  # Gán NaN cho 7 ngày eval theo đúng yêu cầu

cols_train = cols_eval + ['sale_amount_pred']
train_df = pd.read_parquet(timesnet_path, columns=cols_train)

# Ghép nối 90 ngày train + 7 ngày eval
all_data = pd.concat([train_df, eval_df], axis=0, ignore_index=True)
del train_df, eval_df

all_data = all_data.sort_values(by=['store_id', 'product_id', 'dt']).reset_index(drop=True)

# 2. Tạo đặc trưng lịch và thời tiết giống hệt ssa_forecasting.py
dt_series = pd.to_datetime(all_data['dt'])
all_data['time_idx'] = (dt_series - dt_series.min()).dt.days
all_data['dow'] = dt_series.dt.dayofweek
all_data.loc[all_data['dt'].isin(['2024-05-02', '2024-05-03', '2024-05-04']), 'holiday_flag'] = 0
all_data.loc[all_data['holiday_flag'] == 1, 'dow'] = 5
all_data.loc[(all_data['holiday_flag'] == 0) & (all_data['dow'].isin([5, 6])), 'dow'] = 0
all_data['is_precpt'] = (all_data['precpt'] > 3.5).astype(int)

# 3. Đưa sale_amount_pred vào cột thứ 3 của mảng đặc trưng liên tục x_cont
x_cont = all_data[['time_idx', 'discount', 'sale_amount_pred']].values.reshape(-1, 97, 3)
x_cat  = all_data[['dow', 'holiday_flag', 'is_precpt']].values.reshape(-1, 97, 3)

max_encoder_length = 90
holiday_label = 5

date_distance = 18 - x_cont[:, :max_encoder_length, 0][:, None, :] / 5
discount_distance = 1 - np.abs(x_cont[:, max_encoder_length:, 1][..., None] - x_cont[:, :max_encoder_length, 1][:, None, :])

decoder_dow = x_cat[:, max_encoder_length:, 0][..., None]
encoder_dow = x_cat[:, :max_encoder_length, 0][:, None, :]
dow_distance = decoder_dow - encoder_dow

decoder_precpt = x_cat[:, max_encoder_length:, 2][..., None]
encoder_precpt = x_cat[:, :max_encoder_length, 2][:, None, :]
precpt_distance = decoder_precpt - encoder_precpt

date_distance = np.where((decoder_dow == holiday_label).astype(int) - (encoder_dow == holiday_label).astype(int) == 0, date_distance - 3, date_distance)
date_distance = np.where((decoder_dow != holiday_label) & (encoder_dow == holiday_label), date_distance + 3, date_distance)
date_distance = np.where(dow_distance == 0, date_distance - 3, date_distance)
date_distance = np.where((precpt_distance == 0) & (decoder_precpt == 1), date_distance - 3, date_distance)
date_distance = np.where(precpt_distance != 0, date_distance + 3, date_distance)
date_distance = np.clip(date_distance, a_min=1, a_max=18)

weight = (18 - date_distance) * (discount_distance ** 2)
weight = np.exp(weight) / np.exp(weight).sum(axis=-1, keepdims=True)

# 4. Tính psd từ sale_amount GỐC của tập train
mu = all_data[all_data['dt'] <= '2024-06-25'].groupby(['store_id', 'product_id'], as_index=False)['sale_amount'].mean()
mu.rename(columns={'sale_amount': 'psd'}, inplace=True)

pdf = all_data[all_data['dt'] >= '2024-06-26'].copy()
pdf = pdf.merge(mu, on=['store_id', 'product_id'], how='left')

# Dự báo SSA trên chuỗi nhu cầu khôi phục TimesNet
target = x_cont[:, :max_encoder_length, 2:3]
index = encoder_dow.squeeze()[..., None] == np.arange(6)
mean_sale = np.nanmean(np.where(index, target, np.nan), axis=1) + 0.001
season_ratio = np.nanmedian(mean_sale[..., None] / mean_sale[:, None, ...], axis=0)
ratio = season_ratio[decoder_dow, encoder_dow]
pred = (weight * ratio) @ target
pdf['ssa_pred_sale_amount_pred'] = pred.reshape(-1)

# 5. Đánh giá đối chiếu với sale_amount THẬT của eval trên 206.562 dòng hợp lệ
eligible_df = pdf.query('stock_hour6_22_cnt == 0')
eligible_rows = len(eligible_df)
low_sale_rows = len(eligible_df.query('psd < 1'))
high_sale_rows = len(eligible_df.query('psd >= 1'))

print(f"Kiểm tra mẫu hợp lệ: Overall = {eligible_rows:,} | Low Sale = {low_sale_rows:,} | High Sale = {high_sale_rows:,}")
assert eligible_rows == 206562, f"Lỗi: Số dòng hợp lệ là {eligible_rows}, khác 206562!"

def evaluate_group(df_eval_pred, condition):
    wape_list, mae_list, wpe_list = [], [], []
    sub = df_eval_pred.query('stock_hour6_22_cnt == 0').query(condition)
    for _, subdf in sub.groupby('dt'):
        abs_err = (subdf['sale_amount'] - subdf['ssa_pred_sale_amount_pred']).abs()
        wape_list.append(abs_err.sum() / subdf['sale_amount'].sum())
        mae_list.append(abs_err.mean())
        wpe_list.append((subdf['ssa_pred_sale_amount_pred'] - subdf['sale_amount']).sum() / subdf['sale_amount'].sum())
    return round(np.mean(wape_list), 4), round(np.mean(wpe_list), 4), round(np.mean(mae_list), 4)

# Số tham chiếu Bảng 3 bài báo (Cột SSA, dòng TimesNet)
paper_ref_t11 = {
    "Overall":   {"cond": "psd>=0", "wape": 29.79, "wpe": 1.39},
    "Low Sale":  {"cond": "psd<1",  "wape": 37.87, "wpe": 4.89},
    "High Sale": {"cond": "psd>=1", "wape": 23.98, "wpe": -1.12}
}

log_lines = ["group\tdemand\twape\twpe\tmae"]
csv_rows = []

for group_name, info in paper_ref_t11.items():
    wape, wpe, mae = evaluate_group(pdf, info["cond"])
    log_lines.append(f"{info['cond']}\tssa_pred_sale_amount_pred\t{wape}\t{wpe}\t{mae}")

    wape_pct = round(wape * 100, 2)
    wpe_pct = round(wpe * 100, 2)
    csv_rows.append({
        "config_id": "SSA_TimesNet",
        "group": group_name,
        "wape_pct": wape_pct,
        "wpe_pct": wpe_pct,
        "mae": mae,
        "paper_wape_pct": info["wape"],
        "paper_wpe_pct": info["wpe"],
        "delta_wape_pp": round(wape_pct - info["wape"], 2),
        "delta_wpe_pp": round(wpe_pct - info["wpe"], 2),
        "status": "completed"
    })

# 6. Lưu file log, CSV metrics và run_metadata.json
log_content = "\n".join(log_lines) + "\n"
with open("/content/task11_outputs/full/ssa_timesnet.log", "w") as f:
    f.write(log_content)

df_timesnet = pd.DataFrame(csv_rows)
df_timesnet.to_csv("/content/task11_outputs/ssa_timesnet_metrics.csv", index=False)

metadata = {
    "task_id": "Task11",
    "config_id": "SSA_TimesNet",
    "model": "SSA",
    "imputation_source": "TimesNet (Task 10)",
    "demand_timesnet_sha256": sha_timesnet,
    "eval_parquet_sha256": sha_eval,
    "target_column_used": "sale_amount_pred",
    "eligible_eval_rows": eligible_rows,
    "low_sale_eval_rows": low_sale_rows,
    "high_sale_eval_rows": high_sale_rows,
    "execution_timestamp_utc": datetime.now(timezone.utc).isoformat()
}
with open("/content/task11_outputs/run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\n=== NỘI DUNG FILE LOG (ssa_timesnet.log) ===")
print(log_content)
print("=== BẢNG KẾT QUẢ CSV (ssa_timesnet_metrics.csv) ===")
display(df_timesnet)

Kiểm tra mẫu hợp lệ: Overall = 206,562 | Low Sale = 153,208 | High Sale = 53,354

=== NỘI DUNG FILE LOG (ssa_timesnet.log) ===
group	demand	wape	wpe	mae
psd>=0	ssa_pred_sale_amount_pred	0.2979	0.014	0.3709
psd<1	ssa_pred_sale_amount_pred	0.3787	0.0496	0.2656
psd>=1	ssa_pred_sale_amount_pred	0.2398	-0.0115	0.679

=== BẢNG KẾT QUẢ CSV (ssa_timesnet_metrics.csv) ===


,config_id,group,wape_pct,wpe_pct,mae,paper_wape_pct,paper_wpe_pct,delta_wape_pp,delta_wpe_pp,status
0,SSA_TimesNet,Overall,29.79,1.40,0.3709,29.79,1.39,0.0,0.01,completed
1,SSA_TimesNet,Low Sale,37.87,4.96,0.2656,37.87,4.89,0.0,0.07,completed
2,SSA_TimesNet,High Sale,23.98,-1.15,0.6790,23.98,-1.12,0.0,-0.03,completed


In [3]:
import pandas as pd

df_t11 = pd.read_csv("/content/task11_outputs/ssa_timesnet_metrics.csv")

# Số liệu chuẩn 3 nhóm từ Task 8 (SSA_RawSales)
df_t8 = pd.DataFrame([
    {"config_id": "SSA_RawSales", "group": "Overall",   "wape_pct": 31.97, "wpe_pct": -10.50, "mae": 0.3982, "paper_wape_pct": 31.97, "paper_wpe_pct": -10.50, "delta_wape_pp": 0.0, "delta_wpe_pp": 0.0, "status": "completed"},
    {"config_id": "SSA_RawSales", "group": "Low Sale",  "wape_pct": 38.33, "wpe_pct": -8.03,  "mae": 0.2691, "paper_wape_pct": 38.33, "paper_wpe_pct": -8.03,  "delta_wape_pp": 0.0, "delta_wpe_pp": 0.0, "status": "completed"},
    {"config_id": "SSA_RawSales", "group": "High Sale", "wape_pct": 27.40, "wpe_pct": -12.26, "mae": 0.7761, "paper_wape_pct": 27.40, "paper_wpe_pct": -12.26, "delta_wape_pp": 0.0, "delta_wpe_pp": 0.0, "status": "completed"},
])

group_order = {"Overall": 0, "Low Sale": 1, "High Sale": 2}
df_compare = pd.concat([df_t8, df_t11], ignore_index=True)
df_compare["group_rank"] = df_compare["group"].map(group_order)
df_compare = df_compare.sort_values(by=["group_rank", "config_id"]).drop(columns=["group_rank"]).reset_index(drop=True)

compare_path = "/content/task11_outputs/ssa_raw_vs_timesnet_comparison.csv"
df_compare.to_csv(compare_path, index=False)

print("=== BẢNG ĐỐI CHIẾU SONG SONG 6 DÒNG (SSA_RawSales vs SSA_TimesNet) ===")
display(df_compare)

=== BẢNG ĐỐI CHIẾU SONG SONG 6 DÒNG (SSA_RawSales vs SSA_TimesNet) ===


,config_id,group,wape_pct,wpe_pct,mae,paper_wape_pct,paper_wpe_pct,delta_wape_pp,delta_wpe_pp,status
0,SSA_RawSales,Overall,31.97,-10.50,0.3982,31.97,-10.50,0.0,0.00,completed
1,SSA_TimesNet,Overall,29.79,1.40,0.3709,29.79,1.39,0.0,0.01,completed
2,SSA_RawSales,Low Sale,38.33,-8.03,0.2691,38.33,-8.03,0.0,0.00,completed
3,SSA_TimesNet,Low Sale,37.87,4.96,0.2656,37.87,4.89,0.0,0.07,completed
4,SSA_RawSales,High Sale,27.40,-12.26,0.7761,27.40,-12.26,0.0,0.00,completed
5,SSA_TimesNet,High Sale,23.98,-1.15,0.6790,23.98,-1.12,0.0,-0.03,completed
